In [97]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from functions import search_files

In [98]:
bein_src = pd.read_csv( search_files("BEINDATANEWRPT")[0], dtype='str')


In [99]:
bein = bein_src.copy()
bein['End Date'] = pd.to_datetime(bein['End Date'] , dayfirst=True, errors='coerce')
bein['Next Billing Date'] = pd.to_datetime(bein['Next Billing Date'] , dayfirst=True,errors='coerce')
bein = bein.dropna(subset=['End Date','Next Billing Date'])


main_pck_filter = (
            bein["Plan"].str.contains(
                "prem",
                case=False,
                na=False
            )
            |
            bein["Plan"].str.contains(
                "ulti",
                case=False,
                na=False
            )
            |
            bein["Plan"].str.contains(
                "toget",
                case=False,
                na=False
            )
        )


bein = bein.loc[main_pck_filter]
bein = bein.sort_values(['Customer Number','End Date'], ascending=[True,False])

# bein.groupby('Customer Number').agg(count = ("Customer Number",'count')).reset_index()
bein = bein.drop_duplicates(subset=['Customer Number'], keep='first')

In [100]:
bein.loc[bein['Customer Number']=='10000116']

,Customer Number,Customer Type,Entity,Contract Number,Start Date,End Date,Plan,Status,Decoder,Item Description STB,Smart Card,Item Description SC,Next Billing Date,Billing Cycle,PPV Balance,Customer Balance,Outstanding Balance
3074998,10000116,CNE Subscriber,Meet Ghamr,4092109,09-03-2026,2027-09-08,PREMIUM18 Ramadan26 FWC complm.,Active,0302785494,beIN Decoder 1000s,42768429567,beIN Smartcard 1000s,2027-09-09,18M,0 Dr,0 Dr,0 Dr


In [101]:

conditions = [
    # CNE Head office
    bein['Entity'].str.contains('CNE Head office', case=False, na=False)
    | bein['Entity'].str.contains('r2s', case=False, na=False)
    | bein['Entity'].str.contains('cne commercial', case=False, na=False)
    | bein['Entity'].str.contains('FAWRY PLUS', case=False, na=False)
    | bein['Entity'].str.contains('Maintainance', case=False, na=False)
    | bein['Entity'].str.contains('E-Shop', case=False, na=False),
    
    # beIN Head office
    bein['Entity'].str.contains('beIN Head office', case=False, na=False)
    | bein['Entity'].str.contains('bein commercial', case=False, na=False)
    | bein['Entity'].str.contains('beIN Showroom', case=False, na=False),

    # beIN Show Room
    bein['Entity'].str.contains(r'beIN\s*Show', case=False, na=False),

    # CNE Show Room
    bein['Entity'].str.contains('Room', case=False, na=False),

    # beIN Dealers
    bein['Entity'].str.contains('bein', case=False, na=False)

]

choices = [
    'CNE Head office',
    'beIN Head office',
    'beIN Show Room',
    'CNE Show Room',
    'beIN Dealers'
]

bein['Depot'] = np.select(
    conditions,
    choices,
    default='CNE Dealers'
)

In [102]:
bein_cne_dealers = bein.loc[bein['Depot']=='CNE Dealers']

In [103]:
dealers_next_month_ced = bein_cne_dealers.loc[bein_cne_dealers['End Date'].between(pd.to_datetime('2026-10-01'),pd.to_datetime('2026-10-31'))]
list_cne_ced =  dealers_next_month_ced['Entity'].unique()

dealers_next_month_nid = bein_cne_dealers.loc[bein_cne_dealers['Next Billing Date'].between(pd.to_datetime('2026-10-01'),pd.to_datetime('2026-10-31'))]
list_cne_nid =  dealers_next_month_nid['Entity'].unique()

In [104]:
print(dealers_next_month_ced.shape[0])
print(dealers_next_month_ced.drop_duplicates(subset='Customer Number').shape[0])

9430
9430


In [105]:

for file in Path('branches/ced').iterdir():
    if file.is_file():
        file.unlink()

for file in Path('branches/nid').iterdir():
    if file.is_file():
        file.unlink()



In [106]:

for branch in list_cne_ced:
    filename = re.sub(r'[<>:"/\\|?*]', '_', branch)
    dealers_next_month_ced.loc[dealers_next_month_ced['Entity']==branch].drop(columns=['Depot']).to_csv(f'branches/ced/{filename}.csv',index = False)


for branch in list_cne_ced:
    filename = re.sub(r'[<>:"/\\|?*]', '_', branch)
    dealers_next_month_nid.loc[dealers_next_month_nid['Entity']==branch].drop(columns=['Depot']).to_csv(f'branches/nid/{filename}.csv',index = False)